# RL Line Following Robot — MuJoCo Training & Eval
**Colab A100** — full pipeline: install → train straight → train curve → eval videos → download.

Expected total time with 8 parallel envs: ~10 min.

## 1. Install dependencies

In [ ]:
!pip install -q mujoco>=3.1.0 gymnasium>=0.29.0 stable-baselines3>=2.0.0 torch numpy opencv-python-headless
print('Done.')

## 2. Pull repo files from GitHub

In [ ]:
import os, subprocess

REPO = 'https://github.com/aaditey932/rl-line-following-bot.git'
BRANCH = 'colab'

if not os.path.exists('rl-line-following-bot'):
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO], check=True)

os.chdir('rl-line-following-bot')
os.makedirs('models', exist_ok=True)
os.makedirs('eval_videos', exist_ok=True)
print('Working dir:', os.getcwd())
print('Files:', os.listdir('.'))

## 3. Verify setup & benchmark

In [ ]:
import mujoco, gymnasium, stable_baselines3, torch, time
print(f'MuJoCo:    {mujoco.__version__}')
print(f'SB3:       {stable_baselines3.__version__}')
print(f'CUDA:      {torch.cuda.is_available()}')
print(f'CPU cores: {os.cpu_count()}')

from line_follow_env_mujoco import LineFollowEnvMuJoCo
env = LineFollowEnvMuJoCo()
env.reset(seed=0)
t0 = time.perf_counter()
for _ in range(1000):
    env.step(env.action_space.sample())
fps = 1000 / (time.perf_counter() - t0)
env.close()
print(f'Single-env speed: {fps:.0f} fps')
N_ENVS = min(os.cpu_count(), 8)
print(f'Will use {N_ENVS} parallel envs -> ~{fps * N_ENVS:.0f} effective fps')

## 4. Train — Straight track

In [ ]:
!python train_mujoco.py \
  --timesteps 1000000 \
  --save models/ppo_straight \
  --track-type straight \
  --action-delay 1 \
  --motor-dynamics pwm_first_order \
  --n-envs $N_ENVS \
  --quiet
print('Straight model saved.')

## 5. Train — Curve track (scene + domain randomisation)

In [ ]:
!python train_mujoco.py \
  --timesteps 1000000 \
  --save models/ppo_curve \
  --track-type curve \
  --scene-rand \
  --domain-rand \
  --action-delay 1 \
  --motor-dynamics pwm_first_order \
  --n-envs $N_ENVS \
  --quiet
print('Curve model saved.')

## 6. Generate eval videos

In [ ]:
# Straight model on straight track
!python evaluate_mujoco.py \
  --model models/ppo_straight \
  --deterministic --episodes 5 \
  --track-type straight \
  --save-video-root eval_videos
print('Done.')

In [ ]:
# Straight model on curve — generalisation test
!python evaluate_mujoco.py \
  --model models/ppo_straight \
  --deterministic --episodes 5 \
  --track-type curve \
  --save-video-root eval_videos
print('Done.')

In [ ]:
# Curve model on curve track
!python evaluate_mujoco.py \
  --model models/ppo_curve \
  --deterministic --episodes 5 \
  --track-type curve \
  --save-video-root eval_videos
print('Done.')

## 7. Play videos inline

In [ ]:
import glob
from IPython.display import HTML, display
from base64 import b64encode

def play_video(path):
    data = open(path, 'rb').read()
    b64 = b64encode(data).decode()
    display(HTML(f'<p><b>{path}</b></p><video controls width=640><source src="data:video/mp4;base64,{b64}"></video>'))

videos = sorted(glob.glob('eval_videos/**/*.mp4', recursive=True))
print(f'Found {len(videos)} videos')
for v in videos:
    play_video(v)

## 8. Download videos + models

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/eval_videos', 'zip', '.', 'eval_videos')
shutil.make_archive('/content/models', 'zip', '.', 'models')
files.download('/content/eval_videos.zip')
files.download('/content/models.zip')